In [0]:
%run ../init_framework_silver

In [0]:
# 1. SOURCE INGESTION
# Read from Bronze CDF.
df_bronze_loans = read_delta_stream(
    spark=spark, 
    table_name=LOANS_BRONZE
)    

In [0]:
# 2. SCHEMA MAPPING
LOAN_RENAME_MAP = {
    "loan_amnt": "loan_amount",
    "funded_amnt": "funded_amount",
    "term": "loan_term_months",
    "int_rate": "interest_rate",
    "installment": "monthly_installment",
    "issue_d": "issue_date",
    "loan_status": "loan_status",
    "purpose": "loan_purpose"
}

# Align raw bronze names to formal Silver DDL
df_renamed_loans = standardize_column_names(df_bronze_loans, LOAN_RENAME_MAP, strict=True)


In [0]:
# 4. METADATA ENRICHMENT
# Injects standard audit columns (_ingested_at, etc.)
df_with_metadata = apply_silver_metadata(df_renamed_loans)

In [0]:
# 5. DATA QUALITY: CRITICAL NULLS
# Drop records missing fundamental loan attributes; these are non-negotiable for risk modeling.
col_ls = ["loan_amount", "funded_amount", "loan_term_months", "interest_rate", "monthly_installment", "issue_date", "loan_status", "loan_purpose"]
df_valid_loans = drop_critical_nulls(df_with_metadata, col_ls)


In [0]:
# 6. TRANSFORMATION: TERM CONVERSION
from pyspark.sql.functions import col, regexp_replace

# Extract numeric months from string (e.g. "36 months") and convert to annual grain
df_loan_years = df_valid_loans.withColumn(
    "loan_term_years", 
    (regexp_replace(col("loan_term_months"), r"\D+", "").cast("int") / 12).cast("int")
).drop("loan_term_months")


In [0]:
# 7. ENRICHMENT: PURPOSE LOOKUP
from pyspark.sql.functions import col, coalesce, lit, broadcast

# Use a broadcast join for the reference table as it's small (Ref_Loan_Purposes)
df_lkp = spark.read.table(REF_LOAN_PURPOSES).filter("is_active = true")

# Join with aliases to prevent ambiguous column errors
df_joined = (
    df_loan_years.alias("base")
    .join(
        broadcast(df_lkp.alias("lkp")),
        col("base.loan_purpose") == col("lkp.loan_purpose"),
        "left",
    )
    .drop(
        col("base.loan_purpose"),
        col("lkp._updated_at"),
        col("lkp._source_file"),
        col("lkp._processed_by"),
    )
)

# Default unknown or inactive purposes to 'other'
df_loans_final = df_joined.withColumn(
    "loan_purpose", coalesce(col("lkp.loan_purpose"), lit("other"))
).drop("is_active")

In [0]:
# 8. UPSERT LOGIC & STREAM EXECUTION
spark.conf.set("spark.sql.shuffle.partitions", "16")

def upsert_loans_to_silver(microBatchDF, batchId):
    """
    Stateless MERGE into Silver Loans.
    Uses _bronze_version fencing to ensure out-of-order CDF data doesn't corrupt state.
    """
    spark_session = microBatchDF.sparkSession 
    
    from pyspark.sql import Window
    from pyspark.sql.functions import col, row_number, desc
        
    # Filter for valid CDC events (insert or post-image)
    df_updated = microBatchDF.filter(col("_change_type").isin("insert", "update_postimage"))

    # 1. Define the Window
    # Partition by the Primary Key and order by timestamp descending.
    # This ensures the freshest CDC event always receives rank 1.
    window = Window.partitionBy("loan_id").orderBy(col("_updated_at").desc())

    # 2. Add the rank column ONCE
    # This creates a single "Parent" plan for both branches
    df_ranked = df_updated.withColumn("rn", row_number().over(window))

    # --- Branch Logic ---
    # 3. Clean: The absolute latest truth (rn=1) for the Silver Layer MERGE
    df_clean = df_ranked.filter(col("rn") == 1).drop("rn")

    # 4. Bad/Stale: Older intermediate updates (rn>1) for the Audit/Rejected table
    df_bad = df_ranked.filter(col("rn") > 1).drop("rn")

    # --- EXECUTE YOUR WRITES HERE ---    
    # 5. Register Clean Source View for SQL transformations
    df_clean.createOrReplaceTempView("loans_batch_updates")

    merge_query = f"""
    MERGE INTO {LOANS_SILVER} AS target
        USING loans_batch_updates AS source
        ON target.loan_id = source.loan_id
        WHEN MATCHED AND source._bronze_version > target._bronze_version THEN
          UPDATE SET
            target.member_id = source.member_id,
            target.loan_amount = source.loan_amount,
            target.funded_amount = source.funded_amount,
            target.interest_rate = source.interest_rate,
            target.monthly_installment = source.monthly_installment,
            target.issue_date = source.issue_date,
            target.loan_status = source.loan_status,
            target.title = source.title,
            target.loan_term_years = source.loan_term_years,
            target.loan_purpose = source.loan_purpose,
            target._ingested_at = source._ingested_at,
            target._source_file = source._source_file,
            target._bronze_version = source._bronze_version,
            target._updated_at = source._updated_at
        WHEN NOT MATCHED THEN
          INSERT (
            loan_id, member_id, loan_amount, funded_amount, interest_rate, 
            monthly_installment, issue_date, loan_status, title, 
            loan_term_years, loan_purpose, 
            _ingested_at, _source_file, _bronze_version, _updated_at
          )
          VALUES (
            source.loan_id, source.member_id, source.loan_amount, source.funded_amount, source.interest_rate, 
            source.monthly_installment, source.issue_date, source.loan_status, source.title, 
            source.loan_term_years, source.loan_purpose, 
            source._ingested_at, source._source_file, source._bronze_version, source._updated_at
          )
    """
    spark_session.sql(merge_query)


# Execute trigger-once pipeline for batch-like efficiency on streaming logic.
query_loans = (
    df_loans_final.writeStream
    .outputMode("append") 
    .option("checkpointLocation", SILVER_CHECKPOINT_LOANS)
    .trigger(availableNow=True)
    .foreachBatch(upsert_loans_to_silver)
    .start()
)

# block notebook termination until micro-batch is committed
query_loans.awaitTermination()

In [0]:
%sql
select count(1) from lending_risk.silver.loans;
-- select * from lending_risk.silver.loans limit 3;
-- 3334
-- 6674
-- 10014